# 07 — Grad-CAM V2 Explainability

This notebook performs Grad-CAM analysis for the **new 4-class Eye Disease Detection model** trained in `06_model_training_v2.ipynb`.

Classes: **Normal, Cataract, Diabetic Retinopathy, Glaucoma**.

It uses the untouched test split from `preprocessing/splits_v2/test.csv`, the best 4-class EfficientNetV2-B0 model, and the same 224×224 / float32 / 0–255 preprocessing used during training.

In [24]:
# STEP 1 — Imports and reproducibility
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('Working directory:', Path.cwd())

TensorFlow: 2.21.0
Working directory: d:\Practice Projects\Disease Detection\notebooks


In [25]:
# STEP 2 — Project paths and configuration
PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path(r'D:/Practice Projects/Disease Detection')
]
PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'model').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the Disease Detection project root.')

MODEL_PATH = PROJECT_ROOT / 'model' / 'efficientnetv2_b0_4class_best.keras'
METADATA_PATH = PROJECT_ROOT / 'model' / 'model_metadata_v2.json'
TEST_CSV = PROJECT_ROOT / 'preprocessing' / 'splits_v2' / 'test.csv'
OUTPUT_DIR = PROJECT_ROOT / 'reports' / 'gradcam_v2'

CORRECT_DIR = OUTPUT_DIR / 'correct_predictions'
INCORRECT_DIR = OUTPUT_DIR / 'incorrect_predictions'
HIGH_DIR = OUTPUT_DIR / 'high_confidence'
LOW_DIR = OUTPUT_DIR / 'low_confidence'
for directory in [OUTPUT_DIR, CORRECT_DIR, INCORRECT_DIR, HIGH_DIR, LOW_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
CLASS_NAMES = {
    0: 'Normal',
    1: 'Cataract',
    2: 'Diabetic Retinopathy',
    3: 'Glaucoma'
}

MAX_CORRECT_PER_CLASS = 2
MAX_INCORRECT = 8

print('PROJECT_ROOT:', PROJECT_ROOT)
print('MODEL_PATH:', MODEL_PATH)
print('TEST_CSV:', TEST_CSV)
print('OUTPUT_DIR:', OUTPUT_DIR)

PROJECT_ROOT: d:\Practice Projects\Disease Detection
MODEL_PATH: d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
TEST_CSV: d:\Practice Projects\Disease Detection\preprocessing\splits_v2\test.csv
OUTPUT_DIR: d:\Practice Projects\Disease Detection\reports\gradcam_v2


In [26]:
# STEP 3 — Validate required files and load metadata
for required_path in [MODEL_PATH, TEST_CSV]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required file not found: {required_path}')

metadata = {}
if METADATA_PATH.exists():
    with open(METADATA_PATH, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    print('Loaded model_metadata_v2.json')
else:
    print('Warning: model_metadata_v2.json not found; using validated notebook class mapping.')

print('Class mapping:', CLASS_NAMES)
if metadata:
    print('Metadata keys:', list(metadata.keys()))

Loaded model_metadata_v2.json
Class mapping: {0: 'Normal', 1: 'Cataract', 2: 'Diabetic Retinopathy', 3: 'Glaucoma'}
Metadata keys: ['model_name', 'architecture', 'input_size', 'num_classes', 'class_id_to_name', 'preprocessing', 'training_dataset', 'training_config', 'saved_models', 'backbone_layer_name', 'gradcam_target_layer', 'test_metrics']


In [27]:
# STEP 4 — Load the trained 4-class model
model = keras.models.load_model(MODEL_PATH)

print('Model loaded successfully.')
print('Input shape:', model.input_shape)
print('Output shape:', model.output_shape)

if int(model.output_shape[-1]) != 4:
    raise ValueError(f'Expected 4 output classes, found {model.output_shape[-1]}')

print('4-class model validation: PASSED')

Model loaded successfully.
Input shape: (None, 224, 224, 3)
Output shape: (None, 4)
4-class model validation: PASSED


In [28]:
# STEP 5 — Load untouched test split
test_df = pd.read_csv(TEST_CSV)
print('Test rows:', len(test_df))
print('Test columns:', list(test_df.columns))

# Detect common column names used by the dataset-preparation notebook
path_candidates = ['image_path', 'filepath', 'file_path', 'path', 'image']
label_candidates = ['class_id', 'label', 'class', 'target']

IMAGE_COL = next((c for c in path_candidates if c in test_df.columns), None)
LABEL_COL = next((c for c in label_candidates if c in test_df.columns), None)

if IMAGE_COL is None or LABEL_COL is None:
    raise ValueError(f'Could not identify image/label columns. Available columns: {list(test_df.columns)}')

def resolve_image_path(value):
    p = Path(str(value))
    if p.exists():
        return p
    if not p.is_absolute():
        candidate = PROJECT_ROOT / p
        if candidate.exists():
            return candidate
    return p

test_df['resolved_image_path'] = test_df[IMAGE_COL].apply(resolve_image_path)
test_df['true_class_id'] = test_df[LABEL_COL].astype(int)

missing = test_df[~test_df['resolved_image_path'].apply(lambda p: p.exists())]
if len(missing) > 0:
    raise FileNotFoundError(f'{len(missing)} test images could not be resolved. Example: {missing.iloc[0][IMAGE_COL]}')

print('\nTest class distribution:')
print(test_df['true_class_id'].value_counts().sort_index())
print('\nTest split validation: PASSED')

Test rows: 883
Test columns: ['image_path', 'source_dataset', 'original_label', 'disease_name', 'class_id', 'split']

Test class distribution:
true_class_id
0    342
1    156
2    165
3    220
Name: count, dtype: int64

Test split validation: PASSED


In [29]:
# STEP 6 — Exact training-time preprocessing
def load_image_for_model(image_path):
    image = Image.open(image_path).convert('RGB')
    image = image.resize(IMG_SIZE, Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32)
    return np.expand_dims(array, axis=0)

sample_tensor = load_image_for_model(test_df.iloc[0]['resolved_image_path'])
print('Input tensor shape:', sample_tensor.shape)
print('Input dtype:', sample_tensor.dtype)
print('Input range:', float(sample_tensor.min()), 'to', float(sample_tensor.max()))
print('No manual normalization is applied; EfficientNetV2 preprocessing remains inside the model.')

Input tensor shape: (1, 224, 224, 3)
Input dtype: float32
Input range: 0.0 to 255.0
No manual normalization is applied; EfficientNetV2 preprocessing remains inside the model.


In [30]:
# STEP 7 — Identify Grad-CAM target layer (top_activation)
# The model was saved as a flat architecture: all EfficientNetV2-B0 layers
# are direct children of the top-level model (no nested sub-model).
# The correct Grad-CAM target is 'top_activation' — the final conv feature map
# before GlobalAveragePooling2D.

GRADCAM_LAYER = 'top_activation'
target_layer = model.get_layer(GRADCAM_LAYER)
print('Grad-CAM target layer :', target_layer.name)
print('Target output shape   :', target_layer.output.shape)

# feature_shape = (H, W, C) — used for heatmap validation
feature_shape = tuple(target_layer.output.shape[1:])
print('Feature map shape (H,W,C):', feature_shape)

# Build a sub-model: image_input -> top_activation output
# This keeps the computation graph connected so GradientTape can
# compute gradients of the class score w.r.t. the feature maps.
feat_model = keras.Model(
    inputs=model.input,
    outputs=target_layer.output,
    name='feat_extractor',
)
print('Feature extractor model built successfully.')
print('Grad-CAM target validation: PASSED')


Grad-CAM target layer : top_activation
Target output shape   : (None, 7, 7, 1280)
Feature map shape (H,W,C): (7, 7, 1280)
Feature extractor model built successfully.
Grad-CAM target validation: PASSED


In [31]:
# Helper function for Grad-CAM validation

def load_image(image_path, target_size=(224, 224)):
    img = Image.open(image_path).convert("RGB")
    img = img.resize(target_size)
    img_array = np.array(img, dtype=np.float32)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

print("load_image function: READY")

load_image function: READY


In [32]:
# Step 8 - Get Grad-CAM target layer

target_layer = model.get_layer("top_activation")
gap = model.get_layer("gap")
head_bn = model.get_layer("head_bn")
head_dropout = model.get_layer("head_dropout")
disease_output = model.get_layer("disease_output")

print("Target layer      :", target_layer.name)
print("Target output     :", target_layer.output.shape)
print("GAP               :", gap.name)
print("Head BN           :", head_bn.name)
print("Head Dropout      :", head_dropout.name)
print("Disease Output    :", disease_output.name)

Target layer      : top_activation
Target output     : (None, 7, 7, 1280)
GAP               : gap
Head BN           : head_bn
Head Dropout      : head_dropout
Disease Output    : disease_output


In [33]:
# Step 8 - Grad-CAM validation

sample_path = test_df.iloc[0]["image_path"]
sample_image = load_image(sample_path)

# Prediction
preds = model.predict(sample_image, verbose=0)
predicted_class = int(np.argmax(preds[0]))
predicted_confidence = float(preds[0][predicted_class])

print("Predicted class :", CLASS_NAMES[predicted_class])
print("Confidence      :", f"{predicted_confidence:.4f}")

# Get feature maps and calculate gradients
with tf.GradientTape() as tape:
    feature_maps = target_layer.output

    # Create a model that returns target feature maps
    # and final prediction
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[target_layer.output, disease_output.output]
    )

# Re-run through grad_model with tape
with tf.GradientTape() as tape:
    feature_maps, predictions = grad_model(sample_image, training=False)
    class_score = predictions[:, predicted_class]

grads = tape.gradient(class_score, feature_maps)

print("Feature map shape :", feature_maps.shape)
print("Gradient is None  :", grads is None)

if grads is not None:
    print(
        "Gradient range    :",
        float(tf.reduce_min(grads)),
        float(tf.reduce_max(grads))
    )

    # Global average pooling of gradients
    pooled_grads = tf.reduce_mean(grads, axis=(1, 2))

    # Weighted feature maps
    heatmap = tf.reduce_sum(
        feature_maps * pooled_grads[:, tf.newaxis, tf.newaxis, :],
        axis=-1
    )

    # ReLU
    heatmap = tf.nn.relu(heatmap)

    raw_min = float(tf.reduce_min(heatmap))
    raw_max = float(tf.reduce_max(heatmap))

    print("Raw heatmap range :", f"[{raw_min:.6f}, {raw_max:.6f}]")

    if raw_max > 1e-8:
        heatmap = heatmap / raw_max

        print("Normalized range  :",
              f"[{float(tf.reduce_min(heatmap)):.6f}, "
              f"{float(tf.reduce_max(heatmap)):.6f}]")

        print("Grad-CAM validation: PASSED ✅")
    else:
        print("Grad-CAM validation: FAILED ❌ - heatmap is all zero")
else:
    print("Grad-CAM validation: FAILED ❌ - gradients are None")

Predicted class : Cataract
Confidence      : 1.0000


d:\Practice Projects\Disease Detection\.venv311\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['image_input']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


Feature map shape : (1, 7, 7, 1280)
Gradient is None  : False
Gradient range    : -2.0183629203529563e-06 1.914985432449612e-06
Raw heatmap range : [0.000001, 0.000029]
Normalized range  : [0.047324, 1.000000]
Grad-CAM validation: PASSED ✅


In [34]:
# Grad-CAM model components (flat architecture — no nested backbone sub-model)

target_layer = model.get_layer("top_activation")
gap = model.get_layer("gap")
head_bn = model.get_layer("head_bn")
head_dropout = model.get_layer("head_dropout")
disease_output = model.get_layer("disease_output")

grad_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=[target_layer.output, disease_output.output]
)

def make_gradcam_heatmap(input_tensor, class_idx):
    with tf.GradientTape() as tape:
        feature_maps, predictions = grad_model(input_tensor, training=False)
        tape.watch(feature_maps)
        class_score = predictions[:, class_idx]
    grads = tape.gradient(class_score, feature_maps)
    if grads is None:
        raise ValueError('Gradients are None — Grad-CAM failed.')
    pooled_grads = tf.reduce_mean(grads, axis=(1, 2))
    heatmap = tf.reduce_sum(
        feature_maps * pooled_grads[:, tf.newaxis, tf.newaxis, :], axis=-1
    )
    heatmap = tf.nn.relu(heatmap)[0].numpy()
    max_val = heatmap.max()
    if max_val > 1e-8:
        heatmap = heatmap / max_val
    return heatmap

print('Grad-CAM components loaded successfully.')
print('Target layer :', target_layer.name)
print('GAP          :', gap.name)
print('BN           :', head_bn.name)
print('Dropout      :', head_dropout.name)
print('Output       :', disease_output.name)

Grad-CAM components loaded successfully.
Target layer : top_activation
GAP          : gap
BN           : head_bn
Dropout      : head_dropout
Output       : disease_output


In [35]:
# Step 8 - Grad-CAM numerical validation

sample_path = test_df.iloc[0]["image_path"]
sample_image = load_image(sample_path)

preds = model.predict(sample_image, verbose=0)
predicted_class = int(np.argmax(preds[0]))
predicted_confidence = float(preds[0][predicted_class])

print("Predicted class :", CLASS_NAMES[predicted_class])
print("Confidence      :", f"{predicted_confidence:.4f}")

heatmap = make_gradcam_heatmap(sample_image, predicted_class)

print("Heatmap shape :", heatmap.shape)
print("Heatmap range :", f"[{heatmap.min():.6f}, {heatmap.max():.6f}]")

if heatmap.max() > 1e-8:
    print("Grad-CAM validation: PASSED ✅")
else:
    print("Grad-CAM validation: FAILED ❌ - heatmap is all zero")

Predicted class : Cataract
Confidence      : 1.0000
Heatmap shape : (7, 7)
Heatmap range : [0.047324, 1.000000]
Grad-CAM validation: PASSED ✅


In [36]:
# STEP 9 — Prediction + Visualization Helpers

def predict_image(image_path):
    """
    Predict the eye disease for a single retinal image.

    Returns:
        predicted_class: integer class ID
        confidence: prediction confidence
        probabilities: probability for all 4 classes
    """
    tensor = load_image_for_model(image_path)

    probabilities = model.predict(tensor, verbose=0)[0]

    predicted_class = int(np.argmax(probabilities))
    confidence = float(probabilities[predicted_class])

    return predicted_class, confidence, probabilities


def make_visualization(
    image_path,
    true_class,
    predicted_class,
    confidence,
    heatmap,
    save_path,
    title_prefix="Grad-CAM"
):
    """
    Create and save:
    1. Original image
    2. Grad-CAM heatmap
    3. Grad-CAM overlay
    """

    # Load original image
    original = Image.open(image_path).convert("RGB").resize(
        IMG_SIZE,
        Image.Resampling.BILINEAR
    )

    # Convert image to [0, 1]
    base = np.asarray(original, dtype=np.float32) / 255.0

    # Resize Grad-CAM heatmap to original image size
    heatmap_img = Image.fromarray(
        np.uint8(np.clip(heatmap, 0, 1) * 255)
    ).resize(
        IMG_SIZE,
        Image.Resampling.BILINEAR
    )

    heatmap_resized = (
        np.asarray(heatmap_img, dtype=np.float32) / 255.0
    )

    # Convert grayscale heatmap to colored heatmap
    colored = plt.get_cmap("jet")(heatmap_resized)[..., :3]

    # Overlay Grad-CAM on original image
    overlay = np.clip(
        0.60 * base + 0.40 * colored,
        0,
        1
    )

    # Create visualization
    plt.figure(figsize=(15, 5))

    # Original image
    plt.subplot(1, 3, 1)
    plt.imshow(base)
    plt.title(
        f"Original\nTrue: {CLASS_NAMES[true_class]}"
    )
    plt.axis("off")

    # Grad-CAM heatmap
    plt.subplot(1, 3, 2)
    plt.imshow(
        heatmap_resized,
        cmap="jet",
        vmin=0,
        vmax=1
    )
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")

    # Overlay
    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title(
        f"{title_prefix}\n"
        f"Predicted: {CLASS_NAMES[predicted_class]} "
        f"({confidence:.2%})"
    )
    plt.axis("off")

    plt.tight_layout()

    # Save visualization
    plt.savefig(
        save_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()


def analyze_one(row, category, output_dir, rank):
    """
    Analyze one test image:
    - Prediction
    - Confidence
    - Grad-CAM
    - Visualization
    - Result dictionary
    """

    # Support both possible path column names
    image_path = row.get(
        "image_path",
        row.get("resolved_image_path")
    )

    if image_path is None:
        raise KeyError(
            "Row has neither 'image_path' nor "
            "'resolved_image_path'"
        )

    image_path = Path(str(image_path))

    # True class
    true_class = int(row["true_class_id"])

    # Prediction
    predicted_class, confidence, probabilities = predict_image(
        image_path
    )

    # Generate Grad-CAM using the predicted class
    input_tensor = load_image_for_model(image_path)

    heatmap = make_gradcam_heatmap(
        input_tensor,
        predicted_class
    )

    # Validate heatmap shape
    if heatmap.shape != feature_shape[:2]:
        raise ValueError(
            f"Unexpected heatmap shape: {heatmap.shape}; "
            f"expected {feature_shape[:2]}"
        )

    # Validate heatmap values
    if not (
        np.isfinite(heatmap).all()
        and heatmap.min() >= 0
        and heatmap.max() <= 1
    ):
        raise ValueError(
            "Heatmap validation failed: "
            "values must be finite and in [0, 1]."
        )

    # Make sure heatmap is not empty
    if heatmap.max() <= 1e-8:
        raise ValueError(
            "Grad-CAM heatmap is effectively zero."
        )

    # Safe filename
    safe_name = (
        Path(image_path)
        .stem
        .replace(" ", "_")
    )

    # Output path
    save_path = (
        output_dir
        / f"{rank:02d}_{safe_name}_gradcam.png"
    )

    # Create visualization
    make_visualization(
        image_path=image_path,
        true_class=true_class,
        predicted_class=predicted_class,
        confidence=confidence,
        heatmap=heatmap,
        save_path=save_path,
        title_prefix=category.replace("_", " ").title()
    )

    # Final result
    result = {
        "image_path": str(image_path),

        "true_class_id": true_class,
        "true_disease": CLASS_NAMES[true_class],

        "predicted_class_id": predicted_class,
        "predicted_disease": CLASS_NAMES[predicted_class],

        "confidence": confidence,

        "correct_prediction": bool(
            true_class == predicted_class
        ),

        # Correct target layer for current model
        "gradcam_target_layer": "top_activation",

        "heatmap_shape": str(heatmap.shape),

        "heatmap_min": float(heatmap.min()),
        "heatmap_max": float(heatmap.max()),

        "output_path": str(save_path)
    }

    return result


print("Step 9 helpers loaded successfully.")
print("Grad-CAM target layer: top_activation")

Step 9 helpers loaded successfully.
Grad-CAM target layer: top_activation


In [37]:
# STEP 10 — Run prediction over the entire untouched test set

prediction_rows = []

for idx, row in test_df.iterrows():

    # Resolve image path
    image_path = row.get(
        'resolved_image_path',
        row.get('image_path')
    )

    if image_path is None:
        raise KeyError(
            "Test row has neither 'resolved_image_path' nor 'image_path'"
        )

    image_path = Path(str(image_path))

    # True class
    true_class = int(row['true_class_id'])

    # Prediction
    predicted_class, confidence, probabilities = predict_image(
        image_path
    )

    # Store prediction results
    prediction_rows.append({
        'image_path': str(image_path),

        'true_class_id': true_class,
        'true_disease': CLASS_NAMES[true_class],

        'predicted_class_id': predicted_class,
        'predicted_disease': CLASS_NAMES[predicted_class],

        'confidence': confidence,
        'correct_prediction': bool(
            true_class == predicted_class
        ),

        # Individual class probabilities
        'prob_normal': float(probabilities[0]),
        'prob_cataract': float(probabilities[1]),
        'prob_diabetic_retinopathy': float(probabilities[2]),
        'prob_glaucoma': float(probabilities[3])
    })


# Convert results to DataFrame
predictions_df = pd.DataFrame(prediction_rows)


# Save predictions
prediction_output_path = (
    OUTPUT_DIR / 'test_predictions_for_gradcam.csv'
)

predictions_df.to_csv(
    prediction_output_path,
    index=False
)


# Summary
total_images = len(predictions_df)
correct_images = int(
    predictions_df['correct_prediction'].sum()
)
incorrect_images = total_images - correct_images
test_accuracy = (
    correct_images / total_images
    if total_images > 0
    else 0.0
)


print("=" * 60)
print("STEP 10 — TEST SET PREDICTION SUMMARY")
print("=" * 60)

print(f"Test images evaluated : {total_images}")
print(f"Correct predictions   : {correct_images}")
print(f"Incorrect predictions : {incorrect_images}")
print(f"Test accuracy         : {test_accuracy:.2%}")

print("\nPredicted disease distribution:")
print(
    predictions_df['predicted_disease']
    .value_counts()
)

print("\nTrue disease distribution:")
print(
    predictions_df['true_disease']
    .value_counts()
)

print("\nPrediction file saved:")
print(prediction_output_path)

print("=" * 60)

STEP 10 — TEST SET PREDICTION SUMMARY
Test images evaluated : 883
Correct predictions   : 750
Incorrect predictions : 133
Test accuracy         : 84.94%

Predicted disease distribution:
predicted_disease
Normal                  407
Diabetic Retinopathy    172
Cataract                163
Glaucoma                141
Name: count, dtype: int64

True disease distribution:
true_disease
Normal                  342
Glaucoma                220
Diabetic Retinopathy    165
Cataract                156
Name: count, dtype: int64

Prediction file saved:
d:\Practice Projects\Disease Detection\reports\gradcam_v2\test_predictions_for_gradcam.csv


In [38]:
# STEP 11 — Select representative correct predictions, incorrect predictions, and confidence extremes
selected = []

# One or more high-confidence correct examples for each disease
for class_id in range(4):
    candidates = predictions_df[
        (predictions_df['true_class_id'] == class_id) &
        (predictions_df['predicted_class_id'] == class_id)
    ].sort_values('confidence', ascending=False).head(MAX_CORRECT_PER_CLASS)
    for _, row in candidates.iterrows():
        selected.append((row, 'correct_predictions', CORRECT_DIR))

# Highest-confidence incorrect examples (useful for error analysis)
incorrect_df = predictions_df[~predictions_df['correct_prediction']].sort_values('confidence', ascending=False).head(MAX_INCORRECT)
for _, row in incorrect_df.iterrows():
    selected.append((row, 'incorrect_predictions', INCORRECT_DIR))

# Highest and lowest confidence test predictions
highest_row = predictions_df.loc[predictions_df['confidence'].idxmax()]
lowest_row = predictions_df.loc[predictions_df['confidence'].idxmin()]
selected.append((highest_row, 'high_confidence', HIGH_DIR))
selected.append((lowest_row, 'low_confidence', LOW_DIR))

print('Selected Grad-CAM examples:', len(selected))
print('Highest confidence:', highest_row['predicted_disease'], f"{highest_row['confidence']:.2%}")
print('Lowest confidence:', lowest_row['predicted_disease'], f"{lowest_row['confidence']:.2%}")

glaucoma_errors = predictions_df[
    (predictions_df['true_class_id'] == 3) &
    (predictions_df['predicted_class_id'] != 3)
]
print('Glaucoma misclassifications in test set:', len(glaucoma_errors))
print(glaucoma_errors[['true_disease','predicted_disease','confidence']].head(10))

Selected Grad-CAM examples: 18
Highest confidence: Diabetic Retinopathy 100.00%
Lowest confidence: Normal 37.67%
Glaucoma misclassifications in test set: 93
   true_disease predicted_disease  confidence
1      Glaucoma          Cataract    0.565276
4      Glaucoma            Normal    0.498495
19     Glaucoma            Normal    0.739826
20     Glaucoma            Normal    0.678016
31     Glaucoma          Cataract    0.998584
33     Glaucoma            Normal    0.848277
34     Glaucoma            Normal    0.644263
78     Glaucoma            Normal    0.540678
81     Glaucoma            Normal    0.659042
87     Glaucoma            Normal    0.776514


In [39]:
# STEP 12 — Generate and save all selected Grad-CAM visualizations
gradcam_rows = []
category_counters = {}

for row, category, output_dir in selected:
    category_counters[category] = category_counters.get(category, 0) + 1
    result = analyze_one(
        row,
        category,
        output_dir,
        category_counters[category]
    )
    result['category'] = category
    gradcam_rows.append(result)
    print(
        f"{len(gradcam_rows)}/{len(selected)} | "
        f"True: {result['true_disease']} | "
        f"Pred: {result['predicted_disease']} | "
        f"Confidence: {result['confidence']:.2%}"
    )

gradcam_df = pd.DataFrame(gradcam_rows)
gradcam_csv = OUTPUT_DIR / 'gradcam_results.csv'
gradcam_df.to_csv(gradcam_csv, index=False)

print('\nGrad-CAM generation completed.')
print('Results saved:', gradcam_csv)

d:\Practice Projects\Disease Detection\.venv311\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['image_input']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


1/18 | True: Normal | Pred: Normal | Confidence: 99.65%
2/18 | True: Normal | Pred: Normal | Confidence: 99.59%
3/18 | True: Cataract | Pred: Cataract | Confidence: 100.00%
4/18 | True: Cataract | Pred: Cataract | Confidence: 100.00%
5/18 | True: Diabetic Retinopathy | Pred: Diabetic Retinopathy | Confidence: 100.00%
6/18 | True: Diabetic Retinopathy | Pred: Diabetic Retinopathy | Confidence: 100.00%
7/18 | True: Glaucoma | Pred: Glaucoma | Confidence: 99.78%
8/18 | True: Glaucoma | Pred: Glaucoma | Confidence: 99.75%
9/18 | True: Glaucoma | Pred: Cataract | Confidence: 99.86%
10/18 | True: Normal | Pred: Diabetic Retinopathy | Confidence: 99.82%
11/18 | True: Glaucoma | Pred: Cataract | Confidence: 99.60%
12/18 | True: Glaucoma | Pred: Cataract | Confidence: 99.00%
13/18 | True: Glaucoma | Pred: Cataract | Confidence: 97.99%
14/18 | True: Glaucoma | Pred: Normal | Confidence: 97.28%
15/18 | True: Glaucoma | Pred: Diabetic Retinopathy | Confidence: 97.04%
16/18 | True: Glaucoma | Pred:

In [40]:
# STEP 13 — Dedicated glaucoma error analysis
glaucoma_df = predictions_df[predictions_df['true_class_id'] == 3].copy()
glaucoma_df['is_glaucoma_correct'] = glaucoma_df['predicted_class_id'] == 3
glaucoma_df['error_type'] = np.where(
    glaucoma_df['is_glaucoma_correct'],
    'Correct Glaucoma',
    'Glaucoma misclassified as ' + glaucoma_df['predicted_disease']
)

glaucoma_csv = OUTPUT_DIR / 'glaucoma_analysis.csv'
glaucoma_df.to_csv(glaucoma_csv, index=False)

print('Glaucoma test images:', len(glaucoma_df))
print('Correct Glaucoma:', int(glaucoma_df['is_glaucoma_correct'].sum()))
print('Glaucoma recall:', f"{glaucoma_df['is_glaucoma_correct'].mean():.2%}")
print('\nGlaucoma prediction distribution:')
print(glaucoma_df['predicted_disease'].value_counts())
print('\nSaved:', glaucoma_csv)

Glaucoma test images: 220
Correct Glaucoma: 127
Glaucoma recall: 57.73%

Glaucoma prediction distribution:
predicted_disease
Glaucoma                127
Normal                   78
Cataract                 13
Diabetic Retinopathy      2
Name: count, dtype: int64

Saved: d:\Practice Projects\Disease Detection\reports\gradcam_v2\glaucoma_analysis.csv


In [41]:
# STEP 14 — Final summary and output validation
png_files = list(OUTPUT_DIR.rglob('*.png'))

print('=' * 70)
print('GRAD-CAM V2 ANALYSIS COMPLETED')
print('=' * 70)
print(f'Model: {MODEL_PATH}')
print('Classes: Normal | Cataract | Diabetic Retinopathy | Glaucoma')
print('Test images evaluated:', len(predictions_df))
print('Correct predictions:', int(predictions_df['correct_prediction'].sum()))
print('Incorrect predictions:', int((~predictions_df['correct_prediction']).sum()))
print('Test accuracy:', f"{predictions_df['correct_prediction'].mean():.2%}")
print('Glaucoma recall:', f"{glaucoma_df['is_glaucoma_correct'].mean():.2%}")
print('Grad-CAM visualizations generated:', len(png_files))
print('\nMain files:')
print('1.', OUTPUT_DIR / 'gradcam_results.csv')
print('2.', OUTPUT_DIR / 'glaucoma_analysis.csv')
print('3.', OUTPUT_DIR / 'test_predictions_for_gradcam.csv')
print('4.', CORRECT_DIR)
print('5.', INCORRECT_DIR)
print('6.', HIGH_DIR)
print('7.', LOW_DIR)

assert len(CLASS_NAMES) == 4
assert int(model.output_shape[-1]) == 4
assert len(gradcam_df) > 0
print('\nAll final validation checks: PASSED')

GRAD-CAM V2 ANALYSIS COMPLETED
Model: d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
Classes: Normal | Cataract | Diabetic Retinopathy | Glaucoma
Test images evaluated: 883
Correct predictions: 750
Incorrect predictions: 133
Test accuracy: 84.94%
Glaucoma recall: 57.73%
Grad-CAM visualizations generated: 18

Main files:
1. d:\Practice Projects\Disease Detection\reports\gradcam_v2\gradcam_results.csv
2. d:\Practice Projects\Disease Detection\reports\gradcam_v2\glaucoma_analysis.csv
3. d:\Practice Projects\Disease Detection\reports\gradcam_v2\test_predictions_for_gradcam.csv
4. d:\Practice Projects\Disease Detection\reports\gradcam_v2\correct_predictions
5. d:\Practice Projects\Disease Detection\reports\gradcam_v2\incorrect_predictions
6. d:\Practice Projects\Disease Detection\reports\gradcam_v2\high_confidence
7. d:\Practice Projects\Disease Detection\reports\gradcam_v2\low_confidence

All final validation checks: PASSED
